<img src="https://raw.githubusercontent.com/singhpratech/sqljev/main/site/assets/rowl.svg" width="90" align="right">

# Teach Rowl your tables: fine-tune Laya for sqljev

This notebook turns labelled rows from your database into a better model for **one question** you ask in SQL,
for example *"the adverse event was serious"* or *"which team should handle this ticket?"*.

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough), then **Runtime → Run all**.
2. It measures the base Laya model on held-out rows, fine-tunes it (about 10–20 minutes on a T4), and measures again.
3. Optionally it publishes the checkpoint to the Hugging Face Hub, and every database uses it with `SQLJEV_MODEL=<repo>`.

The default settings run a **demo on synthetic adverse-event reports**, a question the base model finds hard
(about 0.70 accuracy zero-shot). Switch `DATA` to your own database or a CSV when you are ready.

Secrets (key icon in the left sidebar), only when you need them: `DB_URL` for your database, `HF_TOKEN` to publish.

In [ ]:
#@title 1. Install sqljev and check the GPU
!pip install -q "sqljev[laya,db]" duckdb pyarrow

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 2. What should the model learn?
DATA = "demo: adverse events (serious?)"  #@param ["demo: adverse events (serious?)", "demo: adverse events (body system)", "demo: company matching", "my database", "CSV upload"]
#@markdown For your own data: a query that returns the columns the model may see **plus** the label column.
SQL = "SELECT drug, narrative, serious FROM adverse_events"  #@param {type:"string"}
LABEL = "serious"  #@param {type:"string"}
#@markdown The question exactly as you will ask it in SQL, e.g. `jev(t, '...')`.
QUESTION = "the adverse event was serious: the patient died, was hospitalised, or it was life-threatening"  #@param {type:"string"}
KIND = "noul"  #@param ["noul", "choice", "score"]
#@markdown For choice/score: comma-separated options (score: lowest first). Blank = the distinct labels.
OPTIONS = ""  #@param {type:"string"}
ROWS = 4000  #@param {type:"integer"}
EPOCHS = 3  #@param {type:"integer"}

In [ ]:
#@title 3. Load labelled rows and split them into train / test
from sqljev.finetune import examples_from_rows, split, write_jsonl

DEMOS = {"demo: adverse events (serious?)": "ae_serious", "demo: adverse events (body system)": "ae_system",
         "demo: company matching": "company_match"}
if DATA in DEMOS:
    from sqljev.demo import TASKS, generate
    name, table, cols, QUESTION, KIND, opts, LABEL = next(t for t in TASKS if t[0] == DEMOS[DATA])
    OPTIONS = ",".join(opts or [])
    rows = [{c: r[c] for c in cols + [LABEL]} for r in generate(ROWS)[table]][:ROWS]
elif DATA == "my database":
    import sqlalchemy as sa
    with sa.create_engine(_secret("DB_URL")).connect() as conn:
        rows = [dict(m) for m in conn.execute(sa.text(SQL)).mappings()][:ROWS]
else:
    import io, pandas as pd
    from google.colab import files
    up = files.upload()
    rows = pd.read_csv(io.BytesIO(next(iter(up.values())))).to_dict("records")[:ROWS]

records = examples_from_rows(rows, QUESTION, KIND, [o.strip() for o in OPTIONS.split(",") if o.strip()] or None, LABEL)
train, test = split(records, test_fraction=0.25)
write_jsonl(train, "train.jsonl"); write_jsonl(test, "test.jsonl")
print("%d labelled rows -> %d train, %d test" % (len(records), len(train), len(test)))
print("the model sees:", records[0]["state"])
print("and is asked:  ", records[0]["questions"]["q"]["instructions"])

In [ ]:
#@title 4. How does the base model do on your held-out rows?
from sqljev.finetune import accuracy
before = accuracy(test, device="cuda")
print("base Laya: %.1f%% correct on %d held-out rows" % (100 * before["accuracy"], before["decisions"]))

In [ ]:
#@title 5. Fine-tune (about 10-20 minutes on a T4)
from sqljev.finetune import finetune
finetune(train, "checkpoint", epochs=EPOCHS)

In [ ]:
#@title 6. Measure again, on the same held-out rows
after = accuracy(test, model="checkpoint", device="cuda")
print("base Laya:       %.1f%%" % (100 * before["accuracy"]))
print("fine-tuned Laya: %.1f%%" % (100 * after["accuracy"]))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 2.2))
ax.barh(["base", "fine-tuned"], [before["accuracy"], after["accuracy"]], color=["#8C9AB8", "#F0A202"])
ax.set_xlim(0, 1); ax.set_xlabel("accuracy on held-out rows")
for i, v in enumerate([before["accuracy"], after["accuracy"]]):
    ax.text(v + 0.01, i, "%.1f%%" % (100 * v), va="center")
plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Try it on a few held-out rows
from sqljev import Jev
jev = Jev(model="checkpoint", device="cuda")
opts = [o.strip() for o in OPTIONS.split(",") if o.strip()] or None
sample = test[:5]
answers = jev.evaluate([r["state"] for r in sample], QUESTION, KIND, opts)
for r, a in zip(sample, answers):
    got = {"noul": lambda: "%.2f" % a["noul"], "choice": lambda: a["choice"], "score": lambda: "%.2f" % a["score"]}[KIND]()
    print("expected %-8s got %-10s %s" % (r["expected"]["q"], got, str(r["state"])[:110]))

In [ ]:
#@title 8. Publish for the team (optional)
#@markdown Uploads the checkpoint to the Hugging Face Hub (needs the `HF_TOKEN` secret with write access).
HF_REPO = ""  #@param {type:"string"}
PUBLIC = False  #@param {type:"boolean"}
if HF_REPO:
    from sqljev.finetune import publish
    url = publish("checkpoint", HF_REPO, token=_secret("HF_TOKEN"), private=not PUBLIC,
                  metrics={"base": before["accuracy"], "finetuned": after["accuracy"], "test_rows": after["decisions"]})
    print("published:", url)
    print("use it everywhere:  SQLJEV_MODEL=%s sqljev gateway --host 0.0.0.0" % HF_REPO)
else:
    print("Set HF_REPO (e.g. your-org/laya-adverse-events) to publish. The checkpoint is in ./checkpoint.")

## Use the new model

The checkpoint answers the question you trained, from any database sqljev supports:

```bash
SQLJEV_MODEL=your-org/laya-adverse-events sqljev gateway --host 0.0.0.0     # SQL Server, Snowflake, BigQuery, Redshift
sqljev judge "$DB_URL" --source adverse_events --prob "..." --model your-org/laya-adverse-events
```
```python
sqljev.spark.register(spark, model="your-org/laya-adverse-events", device="cuda")   # Databricks
sqljev.duckdb.register(con, model="your-org/laya-adverse-events")                     # DuckDB
```

Ask exactly the question you trained: the checkpoint learned that wording. To teach several questions, build
`train.jsonl` from several queries (append the records) and fine-tune once.